# Serie mensual histórica 1981–2023
**Fuente:** Merged Station-Satellite Rainfall – grilla 0.1° (~11 km)  
**Cobertura:** Colombia y países vecinos · 516 meses · 33,464 celdas por mes

In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr

RUTA_NC = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
           r'\Documentos\2025\Outputs\Output4\prcp_mes_acum_1981_2023')

# Listar y ordenar los 516 archivos
archivos = sorted([f for f in os.listdir(RUTA_NC) if f.endswith('.nc')])

# Parsear fecha de cada nombre: rr_YYYYMM.nc
fechas = [pd.Timestamp(int(f[3:7]), int(f[7:9]), 1) for f in archivos]

print(f'Archivos encontrados : {len(archivos)}')
print(f'Período              : {fechas[0].strftime("%Y-%m")} → {fechas[-1].strftime("%Y-%m")}')
print(f'Primer archivo       : {archivos[0]}')
print(f'Último archivo       : {archivos[-1]}')

Archivos encontrados : 516
Período              : 1981-01 → 2023-12
Primer archivo       : rr_198101.nc
Último archivo       : rr_202312.nc


## Construcción del cubo histórico
Apila los 516 archivos en un único objeto 3D: **tiempo × Lat × Lon**  
No hay aproximaciones — cada celda existe igual en todos los archivos.

In [2]:
capas = []

for i, (f, fecha) in enumerate(zip(archivos, fechas)):
    ruta = os.path.join(RUTA_NC, f)
    da = xr.open_dataset(ruta, engine='netcdf4')['prcp']
    # Asignar coordenada de tiempo a esta capa
    da = da.assign_coords(time=fecha).expand_dims('time')
    capas.append(da)
    if (i + 1) % 60 == 0 or i == len(archivos) - 1:
        print(f'  Cargados {i+1}/{len(archivos)}  ({fecha.strftime("%Y-%m")})')

# Concatenar todas las capas a lo largo del eje tiempo
cubo = xr.concat(capas, dim='time')

# Redondear coordenadas para evitar errores de punto flotante
cubo['Lat'] = cubo.Lat.values.round(1)
cubo['Lon'] = cubo.Lon.values.round(1)

print(f'\nCubo construido: {cubo.dims}')
print(f'  Meses : {cubo.sizes["time"]}')
print(f'  Lats  : {cubo.sizes["Lat"]}  ({float(cubo.Lat.min()):.1f}° a {float(cubo.Lat.max()):.1f}°)')
print(f'  Lons  : {cubo.sizes["Lon"]}  ({float(cubo.Lon.min()):.1f}° a {float(cubo.Lon.max()):.1f}°)')
print(f'  Tamaño en memoria: {cubo.nbytes / 1e6:.0f} MB')

  Cargados 60/516  (1985-12)
  Cargados 120/516  (1990-12)
  Cargados 180/516  (1995-12)
  Cargados 240/516  (2000-12)
  Cargados 300/516  (2005-12)
  Cargados 360/516  (2010-12)
  Cargados 420/516  (2015-12)
  Cargados 480/516  (2020-12)
  Cargados 516/516  (2023-12)

Cubo construido: ('time', 'Lat', 'Lon')
  Meses : 516
  Lats  : 188  (-4.6° a 14.1°)
  Lons  : 178  (-82.6° a -64.9°)
  Tamaño en memoria: 69 MB


## Guardar el cubo completo como NetCDF
Un solo archivo con toda la serie histórica — formato estándar, ligero y reutilizable.

In [3]:
SALIDA = os.path.join(RUTA_NC, '..', 'prcp_mensual_1981_2023.nc')
SALIDA = os.path.normpath(SALIDA)

cubo.name = 'prcp'
cubo.attrs = {
    'units': 'mm',
    'long_name': 'Precipitación mensual acumulada',
    'source': 'Merged Station-Satellite Rainfall 0.1deg',
    'periodo': '1981-01 a 2023-12',
}

cubo.to_netcdf(SALIDA)
print(f'Guardado en: {SALIDA}')
print(f'Tamaño archivo: {os.path.getsize(SALIDA) / 1e6:.1f} MB')

Guardado en: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\prcp_mensual_1981_2023.nc
Tamaño archivo: 40.3 MB


---
## Cruce con datos observados de estaciones (marzo–abril 2026)
Carga los registros diarios, agrega a mensual y une con el histórico de la grilla.  
**Nota:** marzo tiene 8 días con dato y abril tiene 22 días — meses incompletos.

In [10]:
import pandas as pd
import numpy as np
import xarray as xr

# ── 1. Cargar datos diarios de estaciones ────────────────────────────────────
RUTA_DIARIA = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
               r'\Documentos\2025\Outputs\Output4\Indicadores'
               r'\precipitacion diaria\precipitacion_diaria.xlsx')

df_diario = pd.read_excel(RUTA_DIARIA)
df_diario['mes'] = df_diario['fecha'].dt.to_period('M')

print(f'Registros diarios cargados: {len(df_diario):,}')
print(f'Estaciones únicas:          {df_diario["codigoestacion"].nunique()}')
print(f'Meses disponibles:          {sorted(df_diario["mes"].unique().astype(str))}')

# ── 2. Agregar diario → mensual por estación ─────────────────────────────────
# sum = precipitación acumulada en los días observados
# count = días con dato ese mes (para saber si el mes está completo)
mensual = (df_diario
    .groupby(['codigoestacion', 'nombreestacion', 'departamento',
              'municipio', 'latitud', 'longitud', 'mes'])
    .agg(
        prcp_mm    = ('precip_acum_diaria', 'sum'),
        dias_dato  = ('precip_acum_diaria', 'count')
    )
    .reset_index()
)

print(f'\nRegistros mensuales: {len(mensual):,}')
print(mensual[['codigoestacion','nombreestacion','mes','prcp_mm','dias_dato']].head(6).to_string())

Registros diarios cargados: 16,295
Estaciones únicas:          649
Meses disponibles:          ['2026-03', '2026-04']

Registros mensuales: 1,275
   codigoestacion nombreestacion      mes  prcp_mm  dias_dato
0        11027030       EL SIETE  2026-03      6.7          8
1        11027030       EL SIETE  2026-04     18.7         22
2        11030010       CERTEGUI  2026-03     77.7          7
3        11030010       CERTEGUI  2026-04    492.7         22
4        11035010          LLORO  2026-03     82.3          8
5        11035010          LLORO  2026-04    498.8         22


In [ ]:
estaciones = mensual[['latitud', 'longitud']].drop_duplicates().reset_index(drop=True)

# Extraer series del cubo en cada estación (método vectorizado con sel nearest)
lat_vals = xr.DataArray(estaciones['latitud'].values, dims='estacion')
lon_vals = xr.DataArray(estaciones['longitud'].values, dims='estacion')

# Selección por vecino más cercano para cada estación
extraido = cubo.sel(Lat=lat_vals, Lon=lon_vals, method='nearest')
# Resultado shape: (time=516, estacion=N)

# Convertir a DataFrame largo
df_extraido = (
    extraido
    .to_dataframe(name='prcp_historico')
    .reset_index()
)

# Unir con las coordenadas reales de las estaciones
df_extraido = df_extraido.merge(
    estaciones.reset_index().rename(columns={'index': 'estacion'}),
    on='estacion',
    suffixes=('_grilla', '_estacion')
)
print(df_extraido.head())
print(df_extraido.shape)

        time  estacion  Lat   Lon  prcp_historico   latitud   longitud
0 1981-01-01         0  5.9 -76.2      597.118408  5.862000 -76.152056
1 1981-01-01         1  5.4 -76.6      607.847656  5.380000 -76.610000
2 1981-01-01         2  5.5 -76.5      575.313110  5.499000 -76.539000
3 1981-01-01         3  5.3 -76.6      587.965881  5.284828 -76.627822
4 1981-01-01         4  5.7 -76.6     1006.532837  5.690556 -76.643778
(334884, 7)


In [36]:
# Renombrar df_extraido para que coincida con df_mensual
df_extraido_clean = (
    df_extraido
    .rename(columns={
        'time':           'mes',
        'prcp_historico': 'prcp_mm',
    })
    .drop(columns=['Lat', 'Lon', 'estacion'])
)

# Agregar columnas que faltan en df_extraido
for col in ['codigoestacion', 'nombreestacion', 'departamento', 'municipio', 'dias_dato']:
    df_extraido_clean[col] = np.nan

# Reordenar columnas igual que mensual
df_extraido_clean = df_extraido_clean[mensual.columns]

# Convertir 'mes' según el dtype de cada DataFrame
# mensual tiene PeriodDtype → convertir a timestamp
if hasattr(mensual['mes'], 'dt') and hasattr(mensual['mes'].dt, 'to_timestamp'):
    mensual['mes'] = mensual['mes'].dt.to_timestamp()

# df_extraido_clean viene de xarray → ya es datetime o string
df_extraido_clean['mes'] = pd.to_datetime(df_extraido_clean['mes'])

# Concatenar
df_total = pd.concat([mensual, df_extraido_clean], ignore_index=True)

print(df_total.shape)
print(df_total['mes'].dtype)

(336159, 9)
datetime64[ns]


In [37]:
df_total.head()

,codigoestacion,nombreestacion,departamento,municipio,latitud,longitud,mes,prcp_mm,dias_dato
0,11027030.0,EL SIETE,CHOCO,EL CARMEN,5.862,-76.152056,2026-03-01,6.7,8.0
1,11027030.0,EL SIETE,CHOCO,EL CARMEN,5.862,-76.152056,2026-04-01,18.7,22.0
2,11030010.0,CERTEGUI,CHOCO,CÉRTEGUI,5.380,-76.610000,2026-03-01,77.7,7.0
3,11030010.0,CERTEGUI,CHOCO,CÉRTEGUI,5.380,-76.610000,2026-04-01,492.7,22.0
4,11035010.0,LLORO,CHOCO,LLORÓ,5.499,-76.539000,2026-03-01,82.3,8.0


In [ ]:
### Tomar la primera estación disponible con coordenadas
### ACA ESTOY VERIFICANDO QUE LA SERIE PARA LA UBICACION CONTEMPLA EN LATITUD Y LONGITUD
### LOS DATOS HISTORICOS Y LOS DIARIOS, PARA PODER HACER UNA COMPARACION VISUAL DE AMBOS

lat_ejemplo = df_total['latitud'].dropna().iloc[0]
lon_ejemplo = df_total['longitud'].dropna().iloc[0]

# Filtrar esa ubicación
serie_unica = (
    df_total[
        (df_total['latitud'] == lat_ejemplo) &
        (df_total['longitud'] == lon_ejemplo)
    ]
    .sort_values('mes')
    [['mes', 'prcp_mm', 'latitud', 'longitud', 'codigoestacion']]
)

print(f"Ubicación: lat={lat_ejemplo}, lon={lon_ejemplo}")
print(f"Registros: {len(serie_unica)}")
print(f"Desde: {serie_unica['mes'].min()}")
print(f"Hasta: {serie_unica['mes'].max()}")
print()
print(serie_unica.head(5))   # primeros de 1981
print("...")
print(serie_unica.tail(5))   # últimos con los de 2026

Ubicación: lat=5.862, lon=-76.15205556
Registros: 518
Desde: 1981-01-01 00:00:00
Hasta: 2026-04-01 00:00:00

            mes      prcp_mm  latitud   longitud  codigoestacion
1275 1981-01-01   597.118408    5.862 -76.152056             NaN
1924 1981-02-01  1001.685608    5.862 -76.152056             NaN
2573 1981-03-01   792.483459    5.862 -76.152056             NaN
3222 1981-04-01  1069.771973    5.862 -76.152056             NaN
3871 1981-05-01  1489.371948    5.862 -76.152056             NaN
...
              mes     prcp_mm  latitud   longitud  codigoestacion
334212 2023-10-01  518.715576    5.862 -76.152056             NaN
334861 2023-11-01  381.025879    5.862 -76.152056             NaN
335510 2023-12-01  190.193649    5.862 -76.152056             NaN
0      2026-03-01    6.700000    5.862 -76.152056      11027030.0
1      2026-04-01   18.700000    5.862 -76.152056      11027030.0
